In [35]:
# Cell 1. 라이브러리 import

from pathlib import Path
import gzip
import re
import json
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from Bio import SeqIO
import mappy as mp

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

In [37]:
# Cell 2. 샘플 및 사용자 설정값

# 1) 비교 분석할 샘플 지정 (fastq.gz 파일 경로)
# 가지고 계신 파일명에 맞게 경로를 수정해 주세요.
SAMPLES = {
    "pDNA_A": Path("pDNA_A_HiFi.fastq.gz"),
    "pDNA_B": Path("pDNA_B_HiFi.fastq.gz")  # PCB 샘플의 실제 파일명으로 수정하세요.
}

# 2) plasmid reference 파일 설정
REFERENCE_FILE = Path("reference.fasta")
REFERENCE_FORMAT = "fasta"   # "fasta" 또는 "genbank"

# 3) 출력 최상위 폴더
OUTDIR = Path("pacbio_polyA_comparative_results")
OUTDIR.mkdir(exist_ok=True, parents=True)

# 4) polyA 위치 및 분석 설정
POLYA_START_1BASED = None
POLYA_END_1BASED = None
POLYA_BASE_IN_REFERENCE = None
MIN_REF_HOMOPOLYMER_LEN = 50
EXPECTED_POLYA_LEN = 80

# 5) Mapping & QC 설정
CIRCULAR_PLASMID = True
ANCHOR_BP = 30
MAPPY_PRESET = "map-hifi"
MAX_HITS_PER_READ = 10
MIN_MAPQ = 0
MAX_NON_POLYA_BASES_IN_REGION_FOR_QC = 5

In [38]:
# Cell 3. Helper 함수들

_RC_TABLE = str.maketrans("ACGTNacgtn", "TGCANtgcan")

def open_text_maybe_gzip(path):
    path = Path(path)
    return gzip.open(path, "rt") if str(path).endswith(".gz") else open(path, "rt")

def revcomp(seq):
    return seq.translate(_RC_TABLE)[::-1]

def load_single_reference(path, fmt="fasta"):
    with open_text_maybe_gzip(path) as handle:
        records = list(SeqIO.parse(handle, fmt))
    if not records:
        raise ValueError(f"Reference record를 찾지 못했습니다: {path}")
    rec = records[0]
    return rec.id, str(rec.seq).upper().replace("U", "T")

def write_fasta(path, name, seq, line_width=80):
    with open(path, "wt") as f:
        f.write(f">{name}\n")
        for i in range(0, len(seq), line_width):
            f.write(seq[i:i+line_width] + "\n")

def find_homopolymer_runs(seq, bases=("A", "T"), min_len=20):
    rows = []
    for base in bases:
        for m in re.finditer(f"{base}+", seq):
            if m.end() - m.start() >= min_len:
                rows.append({
                    "base": base,
                    "start0": m.start(), "end0": m.end(),
                    "length": m.end() - m.start()
                })
    return pd.DataFrame(rows).sort_values(["length", "start0"], ascending=[False, True]).reset_index(drop=True)

def infer_count_base(seq, start0, end0, user_base=None):
    if user_base: return user_base.upper()
    region = seq[start0:end0].upper()
    return "A" if region.count("A") >= region.count("T") else "T"

def parse_cigar(cigar_str):
    if not cigar_str or cigar_str == "*": raise ValueError("No CIGAR string.")
    return [(int(n), op) for n, op in re.findall(r"(\d+)([MIDNSHP=X])", cigar_str)]

def is_reverse_strand(strand):
    return strand.strip() in {"-", "-1"} if isinstance(strand, str) else int(strand) < 0

def longest_run(seq, base):
    max_len = cur = 0
    for b in seq:
        if b == base:
            cur += 1
            max_len = max(max_len, cur)
        else:
            cur = 0
    return max_len

def extract_target_sequence(seq, hit, target_start0, target_end0, count_base):
    seq = seq.upper().replace("U", "T")
    reverse = is_reverse_strand(getattr(hit, "strand", "+"))
    seq_ref_orientation = revcomp(seq) if reverse else seq
    q_pos = len(seq) - int(hit.q_en) if reverse else int(hit.q_st)
    ref_pos = int(hit.r_st)
    
    parts = []
    for length, op in parse_cigar(getattr(hit, "cigar_str", None)):
        if op in {"M", "=", "X"}:
            ov_start, ov_end = max(ref_pos, target_start0), min(ref_pos + length, target_end0)
            if ov_start < ov_end:
                q0 = q_pos + (ov_start - ref_pos)
                parts.append(seq_ref_orientation[q0 : q0 + (ov_end - ov_start)])
            q_pos += length; ref_pos += length
        elif op == "I":
            if target_start0 <= ref_pos <= target_end0:
                parts.append(seq_ref_orientation[q_pos : q_pos + length])
            q_pos += length
        elif op in {"D", "N"}:
            ref_pos += length
        elif op == "S":
            q_pos += length

    region_seq = "".join(parts).upper()
    return {
        "target_seq_ref_orientation": region_seq,
        "polyA_count": region_seq.count(count_base),
        "longest_contiguous_polyA_count": longest_run(region_seq, count_base),
        "non_polyA_base_count": sum(1 for b in region_seq if b != count_base),
    }

def build_mappy_aligner(ref_fasta, preset="map-hifi", best_n=10):
    for p in [preset, "map-pb"]:
        try:
            return mp.Aligner(str(ref_fasta), preset=p, best_n=best_n), p
        except: pass
    raise RuntimeError("mappy aligner 생성 실패")

In [39]:
# Cell 4. Reference 분석 및 매핑 준비

ref_id, ref_seq = load_single_reference(REFERENCE_FILE, REFERENCE_FORMAT)
ref_len = len(ref_seq)
print(f"Reference: {ref_id} ({ref_len:,} bp)")

if POLYA_START_1BASED and POLYA_END_1BASED:
    target_start0, target_end0 = int(POLYA_START_1BASED) - 1, int(POLYA_END_1BASED)
else:
    runs_df = find_homopolymer_runs(ref_seq, min_len=MIN_REF_HOMOPOLYMER_LEN)
    target_start0, target_end0 = int(runs_df.iloc[0]["start0"]), int(runs_df.iloc[0]["end0"])

count_base = infer_count_base(ref_seq, target_start0, target_end0, POLYA_BASE_IN_REFERENCE)
expected_ref_len = target_end0 - target_start0
print(f"Target PolyA/T: {target_start0+1}-{target_end0} (Len: {expected_ref_len}, Base: {count_base})")

analysis_ref_seq = (ref_seq + ref_seq) if CIRCULAR_PLASMID else ref_seq
analysis_ref_file = OUTDIR / "reference_for_mapping.fasta"
write_fasta(analysis_ref_file, ref_id, analysis_ref_seq)

raw_intervals = [(target_start0 + shift, target_end0 + shift) for shift in ([0, ref_len] if CIRCULAR_PLASMID else [0])]
target_intervals = [{"start0": s, "end0": e} for s, e in raw_intervals if s - ANCHOR_BP >= 0 and e + ANCHOR_BP <= len(analysis_ref_seq)]

aligner, actual_preset = build_mappy_aligner(analysis_ref_file, MAPPY_PRESET, MAX_HITS_PER_READ)
print(f"Aligner Ready ({actual_preset})")

Reference: reference (6,473 bp)
Target PolyA/T: 4046-4125 (Len: 80, Base: A)
Aligner Ready (map-hifi)


In [ ]:
# Cell 5. 다중 샘플 데이터 매핑 및 PolyA 추출

all_records = []
qc_records = []
summary_data = []

for sample_name, fastq_path in SAMPLES.items():
    print(f"\n[{sample_name}] Processing {fastq_path}...")
    total_reads = 0
    sample_records = []
    
    for item in tqdm(mp.fastx_read(str(fastq_path)), desc=sample_name, unit="read"):
        read_name, seq = item[0], item[1].upper().replace("U", "T")
        total_reads += 1
        best_row, best_score = None, None

        for hit in aligner.map(seq):
            mapq = getattr(hit, "mapq", 0)
            if mapq < MIN_MAPQ: continue

            for interval in target_intervals:
                t_start, t_end = interval["start0"], interval["end0"]
                if not (hit.r_st <= t_start - ANCHOR_BP and hit.r_en >= t_end + ANCHOR_BP): continue

                try:
                    ext = extract_target_sequence(seq, hit, t_start, t_end, count_base)
                    score = (int(getattr(hit, "is_primary", True)), mapq, getattr(hit, "mlen", 0))
                    
                    row = {
                        "Sample": sample_name,
                        "read_id": read_name,
                        "read_len": len(seq),
                        "hit_mapq": mapq,
                        **ext
                    }
                    if not best_score or score > best_score:
                        best_score, best_row = score, row
                except:
                    pass
        
        if best_row: sample_records.append(best_row)
            
    df = pd.DataFrame(sample_records)
    all_records.append(df)
    
    # QC 필터 적용
    qc_df = df[df["non_polyA_base_count"] <= MAX_NON_POLYA_BASES_IN_REGION_FOR_QC].copy()
    qc_records.append(qc_df)
    
    summary_data.append({
        "Sample": sample_name,
        "Total_Reads": total_reads,
        "Counted_Reads": len(df),
        "QC_Passed_Reads": len(qc_df),
        "Mean_PolyA": qc_df["polyA_count"].mean(),
        "Median_PolyA": qc_df["polyA_count"].median()
    })

combined_df = pd.concat(all_records, ignore_index=True)
combined_qc_df = pd.concat(qc_records, ignore_index=True)
summary_df = pd.DataFrame(summary_data)

# 결과 저장
combined_qc_df.to_csv(OUTDIR / "all_samples_QC_passed.csv", index=False)
summary_df.to_csv(OUTDIR / "all_samples_summary.csv", index=False)

display(summary_df)


[ER200] Processing ER200_HiFi.fastq.gz...


ER200: 0read [00:00, ?read/s]

In [ ]:
# Cell 6. 비교 분석을 위한 Distribution Table 생성

dist_dict = {}

for sample_name in SAMPLES.keys():
    sample_data = combined_qc_df[combined_qc_df["Sample"] == sample_name]["polyA_count"]
    counts = sample_data.value_counts().sort_index()
    
    # 퍼센트 계산
    percentages = (counts / counts.sum()) * 100
    dist_dict[f"{sample_name}_count"] = counts
    dist_dict[f"{sample_name}_percent"] = percentages

# 인덱스(PolyA 길이)를 기준으로 테이블 병합
dist_df = pd.DataFrame(dist_dict).fillna(0).reset_index().rename(columns={"index": "polyA_count"})
dist_df["polyA_count"] = dist_df["polyA_count"].astype(int)

# 분포 데이터 저장
dist_csv = OUTDIR / "comparative_distribution.csv"
dist_df.to_csv(dist_csv, index=False)

print(f"Saved comparative distribution: {dist_csv}")
display(dist_df.head())

In [ ]:
# Cell 7. 두 샘플 간 PolyA 길이 분포 비교 시각화

fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'] # 샘플 추가를 고려한 색상표

for idx, sample_name in enumerate(SAMPLES.keys()):
    col_name = f"{sample_name}_percent"
    ax.plot(dist_df["polyA_count"], dist_df[col_name], 
            label=f"{sample_name} (Median: {summary_df.loc[summary_df['Sample']==sample_name, 'Median_PolyA'].values[0]:.1f})", 
            linewidth=2, marker='o', markersize=4, alpha=0.8, color=colors[idx % len(colors)])
    
    # 영역 채우기 (가독성 향상)
    ax.fill_between(dist_df["polyA_count"], dist_df[col_name], alpha=0.2, color=colors[idx % len(colors)])

ax.axvline(EXPECTED_POLYA_LEN, linestyle="--", color="black", linewidth=1.5, label=f"Expected = {EXPECTED_POLYA_LEN}")

ax.set_xlabel("PolyA Length (Count)", fontsize=12)
ax.set_ylabel("Reads (%)", fontsize=12)
ax.set_title("Comparative PolyA Length Distribution", fontsize=14, pad=15)
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)

# X축 범위 최적화
min_x = max(0, int(dist_df["polyA_count"].min()) - 5)
max_x = int(dist_df["polyA_count"].max()) + 5
ax.set_xlim(min_x, max_x)

plt.tight_layout()

# 비교 차트 저장
plot_path = OUTDIR / "Comparative_PolyA_Distribution.png"
fig.savefig(plot_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved comparison plot: {plot_path}")

In [ ]:
# Cell 8. 샘플별 PolyA Count vs Longest Contiguous Run 비교
# (점들이 대각선 1:1 라인에 모여있을수록 시퀀싱 에러나 Indel이 적은 고품질 PolyA를 의미합니다.)

num_samples = len(SAMPLES)
fig, axes = plt.subplots(1, num_samples, figsize=(6 * num_samples, 6), sharey=True)

# 샘플이 1개일 경우를 대비한 처리
if num_samples == 1: axes = [axes]

for ax, sample_name in zip(axes, SAMPLES.keys()):
    sample_df = combined_qc_df[combined_qc_df["Sample"] == sample_name]
    
    ax.scatter(sample_df["polyA_count"], sample_df["longest_contiguous_polyA_count"], 
               alpha=0.3, edgecolor='none')
    
    # 1:1 기준선
    ax.plot([sample_df["polyA_count"].min(), sample_df["polyA_count"].max()],
            [sample_df["polyA_count"].min(), sample_df["polyA_count"].max()],
            linestyle="--", color="gray", linewidth=1.5)
    
    ax.set_title(f"{sample_name}", fontsize=12)
    ax.set_xlabel("Total PolyA Count")
    if ax == axes[0]: ax.set_ylabel("Longest Contiguous PolyA")
    ax.grid(alpha=0.3)

plt.suptitle("PolyA Quality: Total Count vs Contiguous Run", fontsize=14, y=1.05)
plt.tight_layout()

scatter_path = OUTDIR / "Comparative_PolyA_Scatter.png"
fig.savefig(scatter_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved scatter comparison plot: {scatter_path}")